<a href="https://colab.research.google.com/github/0301sdh/faceguard/blob/main/notebooks/ipadapter_faceid_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IP-Adapter FaceID 보호 효과 실험 (FaceGuard Week 7)

실제로 막고 싶은 위협("내 얼굴이 다른 장면·몸에 들어간 이미지")을 만드는 신원 유지 이미지 생성 도구로 평가 대상을 넓힌다.

**처음 한 번만:** Week 6과 같은 사진을 쓴다 (`MyDrive/faceguard/photos/person_a_1.jpg` ~ `person_a_4.jpg`).

**실행 순서:** 런타임을 T4 GPU로 설정 → `준비 1` 실행 (끝나면 런타임이 자동으로 다시 시작됨) → `준비 2` → `도구 함수` → 실험 셀

**결과 저장:** `MyDrive/faceguard/week7/<tag>/`. FaceNet 채점은 Week 6처럼 Mac에서 한다 (Colab에 facenet-pytorch를 설치하면 PyTorch 버전이 내려가므로).

출처: [IP-Adapter](https://github.com/tencent-ailab/IP-Adapter), [IP-Adapter-FaceID](https://huggingface.co/h94/IP-Adapter-FaceID) (비상업 연구 용도), [insightface](https://github.com/deepinsight/insightface) (모델은 비상업 연구 용도), [diffusers](https://github.com/huggingface/diffusers). 결과와 해석: `docs/weekly/week7.md`

## 준비

In [ ]:
# [준비 1] 패키지 설치. 끝나면 런타임이 자동으로 다시 시작된다.
# - diffusers: Stable Diffusion과 IP-Adapter를 실행하는 라이브러리
# - insightface: FaceID가 신원을 읽을 때 쓰는 얼굴 검출 + ArcFace
# - onnxruntime-gpu: insightface 모델(.onnx 파일)을 GPU로 실행
# Week 5~6 노트북은 SimSwap 때문에 옛 insightface(0.2.1)를 썼지만, 여기서는 FaceID 예시와 같은 최신 버전을 쓴다.
!pip install -q diffusers insightface onnxruntime-gpu

# 새로 설치한 패키지를 쓰려면 런타임을 다시 시작해야 한다. 스스로 종료하면 Colab이 자동으로 다시 시작한다.
import os
os.kill(os.getpid(), 9)

In [ ]:
# [준비 2] 다시 시작된 뒤 실행. Drive 연결 + 모델 불러오기 (처음에는 내려받느라 몇 분 걸림)
import os
import cv2
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')
PHOTO_DIR = '/content/drive/MyDrive/faceguard/photos'
OUT_DIR = '/content/drive/MyDrive/faceguard/week7'

# 1) insightface: FaceID가 신원을 읽는 '목격자'. buffalo_l 안의 검출 모델과 ArcFace만 불러옴
from insightface.app import FaceAnalysis
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition'],
                   providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

# 2) Stable Diffusion 1.5 + IP-Adapter FaceID. 설정은 FaceID 모델 카드 예시를 따름
#    - Realistic Vision: 사진처럼 사실적인 얼굴을 잘 그리도록 조정된 SD 1.5 모델 (VAE는 따로 받음)
#    - DDIM: 노이즈를 빼는 방식(스케줄러). 모델 카드와 같은 값
#    - float16: 메모리를 절반으로 줄여 T4(16GB)에서 여유 있게 돌림
from diffusers import StableDiffusionPipeline, DDIMScheduler, AutoencoderKL
scheduler = DDIMScheduler(num_train_timesteps=1000, beta_start=0.00085, beta_end=0.012,
                          beta_schedule='scaled_linear', clip_sample=False,
                          set_alpha_to_one=False, steps_offset=1)
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse', torch_dtype=torch.float16)
pipe = StableDiffusionPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V4.0_noVAE', scheduler=scheduler, vae=vae,
    torch_dtype=torch.float16, safety_checker=None, feature_extractor=None).to('cuda')

# 3) UNet에 FaceID 통로(cross-attention)를 붙임. FaceID는 CLIP 이미지 인코더를 쓰지 않으므로 image_encoder_folder=None
pipe.load_ip_adapter('h94/IP-Adapter-FaceID', subfolder=None,
                     weight_name='ip-adapter-faceid_sd15.bin', image_encoder_folder=None)
# 얼굴 정보를 얼마나 세게 반영할지 (0~1). 공격자가 가장 닮게 만들려는 상황을 가정해 1.0
pipe.set_ip_adapter_scale(1.0)

# 실행 환경 기록 (week7.md에 남김)
import diffusers, insightface, onnxruntime
print('diffusers', diffusers.__version__, '/ insightface', insightface.__version__,
      '/ onnxruntime', onnxruntime.__version__, onnxruntime.get_device())

## 도구 함수
세션을 새로 열 때마다 실행한다. 함수를 정의만 하므로 바로 끝난다.

In [ ]:
# [도구 함수]
def add_margin(img_bgr, pad=112):
    # 얼굴만 꽉 찬 224×224 사진 둘레에 검은 여백 → 검출 모델이 얼굴을 찾기 쉽게 (Week 6과 같음)
    return cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))

def face_embed(img_bgr):
    # 사진 → insightface 얼굴 검출·정렬 → ArcFace → 길이 1인 512개 숫자. 얼굴을 못 찾으면 None
    # FaceID가 생성 모델에 넘기는 값과 똑같다 (내 사진에서 생성 모델로 가는 유일한 정보)
    faces = app.get(add_margin(img_bgr))
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))   # 여러 개면 가장 큰 얼굴
    return torch.from_numpy(face.normed_embedding)

NEGATIVE = 'monochrome, lowres, bad anatomy, worst quality, low quality, blurry'

def generate(emb, prompt, seed):
    # 얼굴 명세서(512개 숫자) + 프롬프트 → 이미지 (OpenCV BGR)
    # 모양 (1, 1, 1, 512): diffusers 문서의 FaceID 예시와 같은 모양
    pos = emb.view(1, 1, 1, 512)
    # classifier-free guidance: '얼굴 정보 있음'과 '없음(0)' 두 번 예측해서 그 차이를 키움 → 얼굴을 더 강하게 반영
    neg = torch.zeros_like(pos)
    id_embeds = torch.cat([neg, pos]).to('cuda', torch.float16)
    g = torch.Generator('cuda').manual_seed(seed)   # 같은 seed → 같은 출발 노이즈 → 비교 가능
    img = pipe(prompt=prompt, negative_prompt=NEGATIVE, ip_adapter_image_embeds=[id_embeds],
               num_inference_steps=30, width=512, height=512, generator=g).images[0]
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def cos(a, b):
    # 둘 다 길이 1이므로 내적 = 코사인 유사도
    return float((a * b).sum())

## 실험 기록

### 확인: 환경이 돌아가는가 (내 사진 1장 → 이미지 1장)
- 얼굴을 찾고, 이미지가 생성되고, Drive에 저장되면 성공
- 유사도 숫자는 참고만 한다. baseline은 다음 단계에서 사진 4장 × seed 여러 개로 잰다

In [ ]:
os.makedirs(f'{OUT_DIR}/check', exist_ok=True)
src = cv2.imread(f'{PHOTO_DIR}/person_a_1.jpg')
emb = face_embed(src)
print('얼굴 찾음' if emb is not None else '얼굴 못 찾음')

# 얼굴이 너무 작으면 채점할 때 검출이 안 되므로, 처음에는 상반신 장면으로
prompt = 'upper body photo of a person wearing a suit in an office'
out = generate(emb, prompt, seed=0)
cv2.imwrite(f'{OUT_DIR}/check/a_1_seed0.png', out)

out_emb = face_embed(out)
print('생성 결과 vs 내 사진 (insightface ArcFace):',
      '얼굴 못 찾음' if out_emb is None else round(cos(emb, out_emb), 3))

from google.colab.patches import cv2_imshow
cv2_imshow(np.hstack([cv2.resize(src, (512, 512)), out]))